# 演習9 解答編 ―― 安全な終了

> まず `ex09_shutdown.ipynb` を自分で解いてから読んでください。

## 発展課題1 の解答 ―― 番兵が足りないと

**番兵を受け取れなかった Infer が、永久に待ちます。**
そして `join()` が返らないので、`main` も終われません。

順に追うと、こうです。

1. 番兵は1個だけ。Infer0 と Infer1 のうち、**早く `pop()` した1人**が受け取る
2. 受け取った人は `break` して、下流へ番兵を1個流して終わる
3. **もう1人は `q1.pop()` の中で永久に待つ**
4. Show も番兵を2個待っているので、1個しか来ずに永久に待つ
5. `main` の `join()` が返らない

次のセルで確かめます。正しい版（2個）と足りない版（1個）を続けて動かします。
足りない版は止まるので、8秒で強制終了させます。

In [ ]:
%%writefile ans09a.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <atomic>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

// ---- 演習5で組み立てたキュー（ここでは中身は読まなくてよい） ----
template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity) : capacity_(capacity) {}
    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_; });
        q_.push(v); lk.unlock(); can_pop_.notify_one();
    }
    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop(); lk.unlock(); can_push_.notify_one();
        return v;
    }
private:
    std::queue<T> q_;
    std::size_t capacity_;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

const int NWORKER = 2;
const int END = -1;
void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }

void run(int nsentinel) {
    BoundedQueue<int> q1(4), q2(4);
    std::cout << "番兵を " << nsentinel << " 個流す（Infer は " << NWORKER << " 人）\n" << std::flush;

    std::thread reader([&] {
        for (int i = 0; i < 6; i++) { wait_ms(30); q1.push(i); }
        for (int k = 0; k < nsentinel; k++) q1.push(END);
    });
    std::vector<std::thread> inferers;
    for (int k = 0; k < NWORKER; k++)
        inferers.emplace_back([&, k] {
            while (true) {
                int f = q1.pop();                    // ← 番兵が足りないと、ここで永久に待つ
                if (f == END) break;
                wait_ms(60);
                q2.push(f);
            }
            std::cout << "  Infer" << k << " : 終わった\n" << std::flush;
            q2.push(END);
        });
    std::thread shower([&] {
        int seen = 0;
        while (seen < NWORKER) { int f = q2.pop(); if (f == END) seen++; else wait_ms(30); }
        std::cout << "  Show : 終わった\n" << std::flush;
    });
    reader.join();
    for (auto& t : inferers) t.join();
    shower.join();
    std::cout << "  すべて join() できた\n\n" << std::flush;
}

int main() {
    run(NWORKER);      // 正しい
    run(1);            // 足りない -> 止まる
    std::cout << "ここには到達しない\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans09a.cpp -o ans09a
!timeout 8 ./ans09a; echo "終了コード=$? （124 なら止まった）"

足りない版では `Infer1 : 終わった` しか出ていません。Infer0 は待ったままです。

> **番兵方式では「番兵の数 ＝ その段の担当者の数」を必ず合わせる。**

そして、これは**人数を変えるたびに直す必要がある**ということです。
`NWORKER` のような定数1つで両方を書いておかないと、必ず事故ります。

これが番兵方式のいちばんの弱点です。段が増え、段ごとに人数が違ってくると、
「どこに何個流すか」がだんだん追えなくなります。

## 発展課題2 の解答 ―― 複数人のとき、誰が下流を閉じるか

**最後に終わった1人だけ**が閉じます。

```cpp
std::atomic<int> remaining{NWORKER};
...
// 各 Infer の最後に
if (--remaining == 0) q2.close();
```

**2人とも自分が終わったときに呼ぶと、データが消えます。**

理由は単純です。片方が先に終わって `q2.close()` を呼んだ時点で、
**もう片方はまだ処理中**かもしれません。その人が `q2.push(f)` しても、
閉じられているので入りません。**静かに消えます。**

次のセルで確かめます。偶数フレームだけ重い設定にして、片方が先に終わるようにしてあります。

In [ ]:
%%writefile ans09b.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <atomic>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

// ---- 演習5のキューに「閉じる」を足したもの ----
template <typename T>
class ClosableQueue {
public:
    explicit ClosableQueue(std::size_t capacity) : capacity_(capacity) {}

    // 入れる。閉じられていたら何もせず false
    bool push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_ || closed_; });
        if (closed_) return false;
        q_.push(v);
        lk.unlock(); can_pop_.notify_one();
        return true;
    }

    // 取り出す。取れたら true。閉じられていて、かつ空なら false
    bool pop(T& out) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty() || closed_; });
        if (q_.empty()) return false;              // 閉じていて、もう残っていない
        out = q_.front(); q_.pop();
        lk.unlock(); can_push_.notify_one();
        return true;
    }

    // もう入れない、と宣言する
    void close() {
        { std::lock_guard<std::mutex> g(mtx_); closed_ = true; }
        can_pop_.notify_all();                     // 待っている全員を起こす
        can_push_.notify_all();
    }

private:
    std::queue<T> q_;
    std::size_t capacity_;
    bool closed_ = false;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

const int NWORKER = 2;
const int N = 10;
void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }
int infer_ms(int f) { return (f % 2 == 0) ? 100 : 20; }   // 偶数フレームだけ重い

void run(bool count_down) {
    ClosableQueue<int> q1(4), q2(4);
    std::atomic<int> remaining{NWORKER};
    std::atomic<int> lost{0};
    int shown = 0;

    std::thread reader([&] {
        for (int i = 0; i < N; i++) { wait_ms(20); q1.push(i); }
        q1.close();
    });
    std::vector<std::thread> inferers;
    for (int k = 0; k < NWORKER; k++)
        inferers.emplace_back([&, k] {
            int f;
            while (q1.pop(f)) {
                wait_ms(infer_ms(f));
                if (!q2.push(f)) lost++;              // 閉じられていたら入らない
            }
            if (count_down) { if (--remaining == 0) q2.close(); }   // 最後の1人だけ閉じる
            else            { q2.close(); }                         // 終わった人が全員閉じる
        });
    std::thread shower([&] { int f; while (q2.pop(f)) { wait_ms(30); shown++; } });

    reader.join();
    for (auto& t : inferers) t.join();
    shower.join();

    std::cout << (count_down ? "【最後の1人だけが close する】" : "【終わった人が全員 close する】")
              << " 表示できた枚数 = " << shown << " / " << N
              << " 、入れられずに消えた = " << lost << "\n";
}

int main() {
    std::cout << "Infer 2人。偶数フレームは 100ms、奇数は 20ms なので、片方が先に終わる\n\n";
    run(false);
    run(true);
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans09b.cpp -o ans09b && ./ans09b

```
【終わった人が全員 close する】 表示できた枚数 = 9 / 10 、入れられずに消えた = 1
【最後の1人だけが close する】 表示できた枚数 = 10 / 10 、入れられずに消えた = 0
```

**消えたのは1枚だけ**です。ここが怖いところで、
「たまに1枚足りない」という壊れ方は、まず気づけません。
枚数を数えていなければ、永遠に見つかりません。

`--remaining == 0` で判定しているのは、`std::atomic` の
「減らして、その結果を返す」が**1つの不可分な操作**だからです。

```cpp
remaining--;                    // ← これを2人が同時にやると
if (remaining == 0) ...         //    両方が 0 を見る、あるいは両方が見ない
```

と2行に分けて書くと、演習2で見た競合そのものになります。

> **「最後の1人だけがやる」は、`--x == 0` で書く。**

これは終了処理でくり返し出てくる形なので、覚えてしまってください。

## 発展課題3 の解答 ―― 途中でやめる

**`close()` は、どのスレッドから呼んでも構いません。**
鍵で守られているので、キー入力を見ている別のスレッドから呼べます。
ここが番兵方式との決定的な違いです（番兵は列の最後尾に並ぶので、すぐには届きません）。

問題は「残っているデータをどうするか」です。3通りあります。

- **① 上流のキューだけ閉じる** … 残っているものは全部処理してから終わる（きれいな終了）
- **② 全部のキューを閉じる** … 表示は止まる
- **③ 閉じたうえで、残っているものを捨てる** … 本当にすぐ止まる

**②が中途半端になる**ことに注意してください。次のセルで測ります。

In [ ]:
%%writefile ans09c.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

// ---- 演習5のキューに「閉じる」を足したもの ----
template <typename T>
class ClosableQueue {
public:
    explicit ClosableQueue(std::size_t capacity) : capacity_(capacity) {}

    // 入れる。閉じられていたら何もせず false
    bool push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_ || closed_; });
        if (closed_) return false;
        q_.push(v);
        lk.unlock(); can_pop_.notify_one();
        return true;
    }

    // 取り出す。取れたら true。閉じられていて、かつ空なら false
    bool pop(T& out) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty() || closed_; });
        if (q_.empty()) return false;              // 閉じていて、もう残っていない
        out = q_.front(); q_.pop();
        lk.unlock(); can_push_.notify_one();
        return true;
    }

    // もう入れない、と宣言する。discard=true なら、残っているものも捨てる
    void close(bool discard = false) {
        {
            std::lock_guard<std::mutex> g(mtx_);
            closed_ = true;
            if (discard) while (!q_.empty()) q_.pop();
        }
        can_pop_.notify_all();                     // 待っている全員を起こす
        can_push_.notify_all();
    }

private:
    std::queue<T> q_;
    std::size_t capacity_;
    bool closed_ = false;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

enum Mode { DRAIN, CLOSE_ALL, DISCARD };
void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }

void run(Mode m, const char* label) {
    ClosableQueue<int> q1(8), q2(8);
    int shown = 0;
    steady_clock::time_point stopped_at;

    std::thread reader([&] { for (int i = 0; ; i++) { wait_ms(20); if (!q1.push(i)) break; } });
    std::thread inferer([&] { int f; while (q1.pop(f)) { wait_ms(60); q2.push(f); } q2.close(); });
    std::thread shower([&] { int f; while (q2.pop(f)) { wait_ms(30); shown++; } });

    std::thread stopper([&] {                     // 「ユーザが q を押した」係
        wait_ms(1000);
        stopped_at = steady_clock::now();
        if (m == DRAIN)     { q1.close(); }                        // 上流だけ閉じる
        if (m == CLOSE_ALL) { q1.close();       q2.close(); }      // 両方閉じる
        if (m == DISCARD)   { q1.close(true);   q2.close(true); }  // 両方閉じて、残りも捨てる
    });

    reader.join(); inferer.join(); shower.join(); stopper.join();
    int ms = duration_cast<milliseconds>(steady_clock::now() - stopped_at).count();
    std::cout << "  停止指示から終了まで " << ms << " ms 、表示できた枚数 = " << shown
              << "   " << label << "\n";
}

int main() {
    std::cout << "1秒間流したところで「やめる」指示を出す。\n";
    std::cout << "Read 20ms / Infer 60ms / Show 30ms なので、q1 には行列ができている。\n\n";
    run(DRAIN,     "上流だけ閉じる（残りを全部処理してから終わる）");
    run(CLOSE_ALL, "上流と下流を閉じる");
    run(DISCARD,   "閉じて、残っているものも捨てる");
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans09c.cpp -o ans09c && ./ans09c

```
  停止指示から終了まで 560 ms 、表示 25枚   上流だけ閉じる（残りを全部処理してから終わる）
  停止指示から終了まで 524 ms 、表示 16枚   上流と下流を閉じる
  停止指示から終了まで  42 ms 、表示 16枚   閉じて、残っているものも捨てる
```

**②を見てください。表示は止まったのに、終わるまで 524ms もかかっています。**

なぜか。**Infer がまだ `q1` に残っている8枚を処理し続けているから**です。
`q1` を閉じても、`pop()` は残っているものを返します（9-2で見たとおり）。
Infer は **誰も見ない結果を、8枚ぶんまじめに作ってから**終わります。

```
q1 : [ 8枚たまっている ] ──▶ Infer(60ms) ──▶ q2(閉じている) ──▶ Show(終了済み)
       ↑ これを全部処理してからでないと Infer は抜けられない
```

③は残りを捨てるので 42ms で終わります。表示枚数は②と同じ16枚です。
**つまり②は、①のような「全部処理する」でもなく、③のような「すぐ止まる」でもない、
いちばん中途半端な選択**でした。

> **「止める」には2種類ある。**
> **きれいに終える（残りを処理する）か、すぐ止める（残りを捨てる）か。**
> **どちらにするかを決めてから書く。**

用途で言えば、

- **動画ファイルの処理を終える** ⇒ ①。1枚も落としたくない
- **ユーザが `q` を押した** ⇒ ③。待たされるほうが困る
- **エラーで異常終了する** ⇒ ③。結果は使わない

②は、たいてい「①のつもりで書いたら②になっていた」という形で現れます。

## 発展課題4 の解答 ―― `push()` の戻り値を無視すると

**データが静かに消えます。**

```cpp
q2.push(f);        // ← 戻り値を見ていない
```

閉じられていれば `false` が返りますが、無視すれば何事もなかったように次へ進みます。
発展課題2で見た「9枚 / 10枚」が、まさにこれでした。

対処は場面によります。

- **正常な終了の途中なら**、消えて構わないこともあります
  （どうせ下流は終わっている）。**その場合も「意図してそうしている」と書き残す**べきです
- **消えては困るなら**、戻り値を見て、数える／記録する／エラーにする

```cpp
if (!q2.push(f)) { lost++; }        // せめて数える
```

より根本的には、**「閉じたキューに push しようとした」こと自体が設計の不整合**です。
上流が下流より先に閉じられている、ということですから。
発展課題2の `--remaining == 0` のように、
**閉じる順序を上流から下流へ固定する**のが正しい直し方です。

> **戻り値のある関数を作ったなら、無視できないようにするか、無視してよい理由を書く。**

## 発展課題5 の解答 ―― スレッドの中で例外が飛んだら

**そのスレッドを持つプログラム全体が、`std::terminate` で異常終了します。**

スレッド関数から例外が外に出ると、C++ の規則で `std::terminate` が呼ばれます。
`main` の `try/catch` では**受け取れません。** 例外はスレッドをまたがないからです。

```cpp
std::thread t([] {
    throw std::runtime_error("...");     // ← ここで即座にプログラム全体が終わる
});
```

これは「`join()` が返らない」よりはマシに見えるかもしれませんが、
**キューが閉じられないまま終わる**ので、後始末は何もされません。

対処は、**スレッド関数の中で受け止める**ことです。

```cpp
std::thread inferer([&] {
    try {
        int f;
        while (q1.pop(f)) { ...; q2.push(f); }
    } catch (const std::exception& e) {
        std::cerr << "Infer で例外: " << e.what() << "\n";
    }
    q2.close();          // ← 例外が出ても、必ず下流を閉じる
});
```

**`q2.close()` を `try` の外に置いてある**ことが肝心です。
例外で抜けても、下流には終了が伝わります。

> **例外で抜ける道も、正常に抜ける道と同じ後始末を通らなければならない。**

これは演習3で見た **RAII** と同じ考え方です。
`lock_guard` が例外で抜けても必ず解錠してくれるのと同じで、
終了処理も「どの抜け方でも必ず通る場所」に置きます。
（クラスのデストラクタで `close()` を呼ぶようにしておくのが、いちばん確実です。）

## 発展課題6 の解答 ―― `detach()` では解決しない

`detach()` は「もう面倒を見ない」という宣言です。
確かに `join()` を呼ばなくてよくなり、9-1 のプログラムは**終わるように見えます。**

**しかし、何も解決していません。**

- **スレッドは動き続けています。** `pop()` の中で待ったままです
- `main` が `return` すると、**まだ動いているスレッドがあるのに**
  プログラムが終わります。その瞬間に何が起きるかは決まっていません
- キューやフレームのメモリが解放されている最中に、そのスレッドが触るかもしれません
- そして、**終わっていない仕事があっても気づけません**。
  9-1 で「10枚処理してから止まった」ことが分かったのは、`join()` が返らなかったからです

> **`join()` が返らないのは、バグを教えてくれているということ。**
> **`detach()` はその警告を消すだけで、バグは残る。**

`join()` が返らないときにやるべきことは、`detach()` に書き換えることではなく、
**なぜ終了が伝わっていないのかを探すこと**です。
まとめのチェックリスト①〜④を上から順に確かめてください。

---

## 参考：本番のプログラムでは

パイプラインを改造したあと、**`Ctrl+C` を押さないと止まらない**、
あるいは**動画が終わっても終了しない**、という状態になることがあります。

そうなったら、このチェックリストを上から順に見てください。

1. 増やしたスレッドに、抜ける道はありますか
2. 増やしたキューの終了は、下流に伝わっていますか
3. 担当者を2人に増やしたなら、**下流を閉じるのは最後の1人だけ**になっていますか
4. 番兵方式なら、**番兵の数は担当者の数と合っていますか**

3番と4番は、**演習7で担当者を増やした瞬間に壊れる**ところです。
「速くしたら終わらなくなった」は、この2つのどちらかであることがほとんどです。